In [1]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.pt') or f.endswith('.mp4'):
            print(os.path.join(root, f))

/kaggle/input/datasets/zlemdemir/model-files/operator_shadowing_best.pt
/kaggle/input/datasets/zlemdemir/3-person-video/video_for_tracking_2.mp4


In [2]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.4 MB/s eta 0:00:00


In [3]:
from ultralytics import YOLO
import pandas as pd


model = YOLO('/kaggle/input/datasets/zlemdemir/model-files/operator_shadowing_best.pt')


results = model.track(
    source='/kaggle/input/datasets/zlemdemir/3-person-video/video_for_tracking_2.mp4',
    tracker='bytetrack.yaml',
    save=True,
    conf=0.5,
    project='/kaggle/working',
    name='koordinat_kaydi'
)


koordinatlar = []

for frame_idx, result in enumerate(results):
    if result.boxes.id is not None:  
        boxes = result.boxes.xywh.cpu().numpy()  # x_merkez, y_merkez, genişlik, yükseklik
        ids = result.boxes.id.cpu().numpy().astype(int)
        classes = result.boxes.cls.cpu().numpy().astype(int)
        
        for box, track_id, cls in zip(boxes, ids, classes):
            x_center, y_center, w, h = box
            koordinatlar.append({
                'frame': frame_idx,
                'id': int(track_id),
                'class': 'person' if cls == 0 else 'high-vis',
                'x': float(x_center),
                'y': float(y_center),
                'width': float(w),
                'height': float(h)
            })

# csv olarak koordinatları kaydettik
df = pd.DataFrame(koordinatlar)
df.to_csv('/kaggle/working/koordinatlar.csv', index=False)

print(f"Toplam {len(df)} koordinat kaydedildi.")
print(f"\nBenzersiz ID sayısı: {df['id'].nunique()}")
print(f"Toplam frame sayısı: {df['frame'].max() + 1}")
print("\nİlk 10 satır:")
print(df.head(10))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 337ms
Prepared 1 package in 39ms
Installed 1 package in 6ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r 

In [4]:
print("Her ID'nin görünme sayısı:")
print(df.groupby(['id', 'class']).size().reset_index(name='count'))

Her ID'nin görünme sayısı:
   id   class  count
0   1  person    185
1   2  person    152
2   3  person    154
3   5  person      2
4   6  person      1


In [5]:
# 4 ve 5 id li personlar ID kaybolmasından dolayı olmus, onları siliyoruz
gecerli_idler = df.groupby('id').size()
gecerli_idler = gecerli_idler[gecerli_idler >= 5].index
df_temiz = df[df['id'].isin(gecerli_idler)]
df_temiz.to_csv('/kaggle/working/koordinatlar_temiz.csv', index=False)
print(f"Temizleme sonrası: {len(df_temiz)} koordinat, {df_temiz['id'].nunique()} ID")

Temizleme sonrası: 491 koordinat, 3 ID
